In [2]:
!pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable


In [22]:
import kagglehub

# Download latest version
file_path = kagglehub.dataset_download("willyard/spam-email-dataset")

print("Path to dataset files:", file_path)

Path to dataset files: /home/julia/.cache/kagglehub/datasets/willyard/spam-email-dataset/versions/1


In [ ]:
import pandas as pd
import os

path = "/home/julia/.cache/kagglehub/datasets/willyard/spam-email-dataset/versions/1"
file_path = os.path.join(path, "enron_spam_data.csv")

# Загружаем данные и смотрим на структуру
df = pd.read_csv(file_path)
print("Структура данных:")
print(df.info())
print("\nПервые 5 строк:")
print(df.head())
print("\nКолонки:")
print(df.columns.tolist())
print("\nУникальные значения в колонке Spam/Ham:")
print(df["Spam/Ham"].value_counts())
print("\nПримеры данных:")
print(df[["Message", "Spam/Ham"]].head(10))

Структура данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  33716 non-null  int64 
 1   Subject     33716 non-null  object
 2   Message     33664 non-null  object
 3   Spam/Ham    33716 non-null  object
 4   Date        33716 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.3+ MB
None

Первые 5 строк:
   Unnamed: 0                       Subject  \
0           0  christmas tree farm pictures   
1           1      vastar resources , inc .   
2           2  calpine daily gas nomination   
3           3                    re : issue   
4           4     meter 7268 nov allocation   

                                             Message Spam/Ham        Date  
0                                                NaN      ham  1999-12-10  
1  gary , production from the high island larger ...      ham  1999-12-13  
2          

In [6]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 4.8 MB/s eta 0:00:0000:0100:01


In [2]:
import pandas as pd
import os
import re
import math
from collections import defaultdict
from sklearn.model_selection import train_test_split
import numpy as np


class NaiveBayesSpamClassifier:
    def __init__(self):
        self.spam_words = defaultdict(int)
        self.ham_words = defaultdict(int)
        self.spam_count = 0
        self.ham_count = 0
        self.total_count = 0
        self.vocabulary = set()

    def preprocess_text(self, text):
        """Преобразует текст в множество слов в нижнем регистре"""
        if pd.isna(text) or text == "":
            return set()
        words = re.findall(r"\b[a-zA-Z]{2,}\b", str(text).lower())
        return set(words)

    def train(self, emails, labels):
        """Обучает модель на размеченных данных"""
        for email, label in zip(emails, labels):
            words = self.preprocess_text(email)
            if not words:
                continue

            self.vocabulary.update(words)

            if label == "spam":
                self.spam_count += 1
                for word in words:
                    self.spam_words[word] += 1
            else:  # 'ham'
                self.ham_count += 1
                for word in words:
                    self.ham_words[word] += 1

        self.total_count = self.spam_count + self.ham_count

    def calculate_probability(self, word, is_spam):
        """Вычисляет вероятность слова с применением сглаживания Лапласа"""
        if is_spam:
            word_count = self.spam_words.get(word, 0)
            total_words = self.spam_count
        else:
            word_count = self.ham_words.get(word, 0)
            total_words = self.ham_count

        return (word_count + 1) / (total_words + 2)

    def predict(self, email):
        """Классифицирует email как спам или не спам
        P(спам|используются слова ...) > 0.5
        """
        words = self.preprocess_text(email)
        if not words:
            return "ham"

        p_spam = self.spam_count / self.total_count
        p_ham = self.ham_count / self.total_count
        
        log_sum_spam = math.log(p_spam)
        log_sum_ham = math.log(p_ham)

        for word in words:
            p_word_spam = self.calculate_probability(word, True)
            p_word_ham = self.calculate_probability(word, False)

            log_sum_spam += math.log(p_word_spam)
            log_sum_ham += math.log(p_word_ham)

        return "spam" if log_sum_spam > log_sum_ham else "ham"

    def evaluate(self, test_emails, test_labels):
        """Оценивает качество классификатора на тестовой выборке"""
        correct = 0
        true_spam = 0
        predicted_spam_correct = 0
        true_ham = 0
        predicted_ham_correct = 0

        predictions = []
        actual_vs_predicted = []

        print("Начинаем оценку на тестовой выборке...")
        for i, (email, true_label) in enumerate(zip(test_emails, test_labels)):
            predicted_label = self.predict(email)
            predictions.append(predicted_label)
            actual_vs_predicted.append((true_label, predicted_label))

            if predicted_label == true_label:
                correct += 1

            if true_label == "spam":
                true_spam += 1
                if predicted_label == "spam":
                    predicted_spam_correct += 1
            else:  # 'ham'
                true_ham += 1
                if predicted_label == "ham":
                    predicted_ham_correct += 1

        accuracy = correct / len(test_emails)
        sensitivity = predicted_spam_correct / true_spam if true_spam > 0 else 0
        specificity = predicted_ham_correct / true_ham if true_ham > 0 else 0

        confusion_matrix = {
            "TP": predicted_spam_correct,
            "FP": true_ham - predicted_ham_correct,
            "TN": predicted_ham_correct,
            "FN": true_spam - predicted_spam_correct,
        }

        return {
            "accuracy": accuracy,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "confusion_matrix": confusion_matrix,
            "predictions": predictions,
            "actual_vs_predicted": actual_vs_predicted,
        }


def main():
    path = ("/home/julia/.cache/kagglehub/datasets/willyard/spam-email-dataset/versions/1")
    file_path = os.path.join(path, "enron_spam_data.csv")
    df = pd.read_csv(file_path)

    # Очищаем данные
    data = df[["Message", "Spam/Ham"]].copy()
    data = data.dropna(subset=["Message"])
    data = data[data["Message"].str.strip() != ""]
    data["Spam/Ham"] = data["Spam/Ham"].str.lower()

    print(f"\nПосле очистки:")
    print(f"Всего записей: {len(data)}")
    print(f"Спам: {len(data[data['Spam/Ham'] == 'spam'])}")
    print(f"Хам: {len(data[data['Spam/Ham'] == 'ham'])}")

    X = data["Message"].tolist()
    y = data["Spam/Ham"].tolist()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"\n=== ОБУЧЕНИЕ МОДЕЛИ ===")
    classifier = NaiveBayesSpamClassifier()
    classifier.train(X_train, y_train) 
    print(f"\n=== ОЦЕНКА МОДЕЛИ НА ТЕСТОВОЙ ВЫБОРКЕ ===")
    results = classifier.evaluate(X_test, y_test) 

    print(f"Результаты на тестовой выборке:")
    print(f"Точность: {results['accuracy']:.4f} ({results['accuracy']*100:.2f}%)")
    print(f"Чувствительность (полнота): {results['sensitivity']:.4f}")
    print(f"Специфичность: {results['specificity']:.4f}")

    cm = results["confusion_matrix"]
    print(f"\nМатрица ошибок:")
    print(f"True Positive (TP): {cm['TP']} - правильно распознанный спам")
    print(f"False Positive (FP): {cm['FP']} - хам, ошибочно помеченный как спам")
    print(f"True Negative (TN): {cm['TN']} - правильно распознанный хам")
    print(f"False Negative (FN): {cm['FN']} - спам, ошибочно помеченный как хам")


if __name__ == "__main__":
    main()


После очистки:
Всего записей: 33345
Спам: 16852
Хам: 16493

=== ОБУЧЕНИЕ МОДЕЛИ ===

=== ОЦЕНКА МОДЕЛИ НА ТЕСТОВОЙ ВЫБОРКЕ ===
Начинаем оценку на тестовой выборке...
Результаты на тестовой выборке:
Точность: 0.9804 (98.04%)
Чувствительность (полнота): 0.9682
Специфичность: 0.9927

Матрица ошибок:
True Positive (TP): 3263 - правильно распознанный спам
False Positive (FP): 24 - хам, ошибочно помеченный как спам
True Negative (TN): 3275 - правильно распознанный хам
False Negative (FN): 107 - спам, ошибочно помеченный как хам
